# Neighborhood Filter

A **neighborhood filter** replaces the value at each grid element with the result
of a user-specified function applied to all grid elements whose centers fall within
a circular neighborhood of radius `r` degrees around that element.

Unlike a fixed *k*-nearest-neighbor average, a radius-based filter imposes a
consistent spatial scale across the whole mesh—useful for variable-resolution grids
where the number of neighbors varies from region to region.

**Supported element types:** face-centered, node-centered, and edge-centered data.

**API at a glance:**

| Object | Method |
|---|---|
| `UxDataArray` | `da.neighborhood_filter(func=np.mean, r=5.0)` |
| `UxDataset` | `ds.neighborhood_filter(func=np.mean, r=5.0)` |

The returned object is always the same type as the input, with the same grid, dims,
coordinates, name, and attributes preserved.


## Imports

In [ ]:
from functools import partial

import numpy as np

import uxarray as ux

## Load Sample Data

We use the `outCSne30-vortex` tutorial dataset (a cubed-sphere grid with 5,400
faces and a synthetic vortex field `psi`).

In [ ]:
uxds = ux.tutorial.open_dataset("outCSne30-vortex")
uxda = uxds["psi"]
uxda

## Visualize the Unfiltered Field

In [ ]:
uxda.plot.polygons(
    cmap="RdBu_r",
    title="Original field (psi)",
    width=700,
    height=400,
)

## Basic Usage: Mean Filter

Calling `neighborhood_filter` with `func=np.mean` and a radius of 5° replaces
each face value with the mean of all face centers within 5° of that face's center.


In [ ]:
uxda_smooth = uxda.neighborhood_filter(func=np.mean, r=5.0)
uxda_smooth

Note that the output is a `UxDataArray` mapped to the same grid and with the same
dimensions as the input. The name, attributes, and coordinates are preserved.


In [ ]:
uxda_smooth.plot.polygons(
    cmap="RdBu_r",
    title="Mean filter (r = 5°)",
    width=700,
    height=400,
)

### Effect of Radius

Increasing `r` produces stronger smoothing. A radius of 0° recovers the original
field (the only element in any neighborhood is the element itself).


In [ ]:
import holoviews as hv
hv.extension("bokeh")

plots = [
    uxda.neighborhood_filter(func=np.mean, r=r).plot.polygons(
        cmap="RdBu_r",
        title=f"r = {r}°",
        width=350,
        height=250,
        clim=(uxda.values.min(), uxda.values.max()),
    )
    for r in [0.0, 2.5, 5.0, 10.0]
]

(plots[0] + plots[1] + plots[2] + plots[3]).cols(2)

## Custom Functions via `functools.partial`

Any callable that accepts an `axis` keyword argument (as NumPy reductions do) works
as the filter function. Use `functools.partial` to fix additional keyword arguments.


In [ ]:
# 90th-percentile filter — highlights local maxima
uxda_p90 = uxda.neighborhood_filter(func=partial(np.percentile, q=90), r=5.0)

# Maximum filter
uxda_max = uxda.neighborhood_filter(func=np.max, r=5.0)

# Median filter — robust to outliers
uxda_med = uxda.neighborhood_filter(func=np.median, r=5.0)

print("max filter max :", uxda_max.values.max())
print("p90 filter max  :", uxda_p90.values.max())
print("median filter max:", uxda_med.values.max())

In [ ]:
(
    uxda_max.plot.polygons(cmap="RdBu_r", title="Max filter (r=5°)", width=350, height=250)
    + uxda_med.plot.polygons(cmap="RdBu_r", title="Median filter (r=5°)", width=350, height=250)
).cols(2)

## Node- and Edge-Centered Data

`neighborhood_filter` works for any data element type. Here we create
synthetic node- and edge-centered fields on a HEALPix grid and filter them.


In [ ]:
uxgrid = ux.Grid.from_healpix(zoom=3)  # 768 faces, 770 nodes, 1536 edges

# Node-centered: a gradient along longitude
node_da = ux.UxDataArray(
    uxgrid.node_lon.values,
    dims=["n_node"],
    uxgrid=uxgrid,
    name="node_lon",
    attrs={"units": "degrees_east"},
)

filtered_node = node_da.neighborhood_filter(func=np.mean, r=10.0)
print("node input  dims:", node_da.dims, "  shape:", node_da.shape)
print("node output dims:", filtered_node.dims, "  shape:", filtered_node.shape)
print("attrs preserved:", filtered_node.attrs)

In [ ]:
# Edge-centered: a random field
rng = np.random.default_rng(42)
edge_da = ux.UxDataArray(
    rng.standard_normal(uxgrid.n_edge),
    dims=["n_edge"],
    uxgrid=uxgrid,
    name="edge_noise",
)

filtered_edge = edge_da.neighborhood_filter(func=np.mean, r=10.0)
print("edge input  dims:", edge_da.dims, "  shape:", edge_da.shape)
print("edge output dims:", filtered_edge.dims, "  shape:", filtered_edge.shape)

## Multi-Dimensional Data (e.g. Time + Space)

When a `UxDataArray` has extra leading dimensions (e.g. `time`), `neighborhood_filter`
applies the spatial filter independently at each time step and preserves the full
dimension order.


In [ ]:
uxds_ts = ux.tutorial.open_dataset("outCSne30-timeseries")
uxda_ts = uxds_ts["psi"]

print("Input  dims:", uxda_ts.dims, " shape:", uxda_ts.shape)

filtered_ts = uxda_ts.neighborhood_filter(func=np.mean, r=5.0)

print("Output dims:", filtered_ts.dims, " shape:", filtered_ts.shape)

The grid and time dimensions are both preserved.  Because the filter is applied
per time step, memory usage scales with `n_time × n_face` as expected.


## Dataset-Level Usage

`UxDataset.neighborhood_filter` applies the filter to **every data variable**
that is mapped to a grid element. Variables without a grid dimension (e.g. scalars
or time-only arrays) are passed through unchanged.


In [ ]:
uxds_filtered = uxds.neighborhood_filter(func=np.mean, r=5.0)
uxds_filtered

## Chaining with xarray Operations

Because `neighborhood_filter` returns a proper `UxDataArray` with its `uxgrid`
preserved, you can chain it with any standard xarray operation.


In [ ]:
# Apply the filter and then mask values below zero
result = (
    uxda
    .neighborhood_filter(func=np.mean, r=5.0)
    .where(lambda x: x > 0)
)
print("Masked result type:", type(result).__name__)
print("uxgrid preserved:", result.uxgrid is not None)
print("Positive fraction:", float((result > 0).sum()) / result.size)

In [ ]:
# Group by latitude band after smoothing (standard xarray groupby)
import xarray as xr

lat_bins = xr.DataArray(
    np.digitize(uxda.uxgrid.face_lat.values, bins=np.arange(-90, 91, 30)),
    dims=["n_face"],
)

zonal_smooth = (
    uxda
    .neighborhood_filter(func=np.mean, r=5.0)
    .groupby(lat_bins)
    .mean()
)
print("Grouped result type:", type(zonal_smooth).__name__)
print("Zonal means:", zonal_smooth.values)

## Empty Neighborhoods and NaN Behavior

If the search radius is so small that *no* neighbor is found for a given element,
the result for that element is `NaN` rather than an uninitialized garbage value.
In practice this only happens for extremely small radii on very coarse grids; the
filter always includes the queried element itself, so `r = 0` is safe.


In [ ]:
uxgrid_coarse = ux.Grid.from_healpix(zoom=1)  # 48 faces
da_coarse = ux.UxDataArray(
    np.arange(uxgrid_coarse.n_face, dtype=float),
    dims=["n_face"],
    uxgrid=uxgrid_coarse,
)

# r = 0 always catches at least the element itself → no NaNs
filtered_r0 = da_coarse.neighborhood_filter(func=np.mean, r=0.0)
print("r = 0:   NaN count =", int(np.isnan(filtered_r0.values).sum()))

# r = 360 catches every element → all values equal the global mean
filtered_global = da_coarse.neighborhood_filter(func=np.mean, r=360.0)
print("r = 360: all equal global mean?",
      np.allclose(filtered_global.values, da_coarse.values.mean()))

## API Reference

See also:

- {py:meth}`uxarray.UxDataArray.neighborhood_filter`
- {py:meth}`uxarray.UxDataset.neighborhood_filter`

Related methods that apply aggregations across different grid element types:

- {py:meth}`uxarray.UxDataArray.topological_mean` — aggregate node→face, node→edge, etc.
- {py:meth}`uxarray.UxDataArray.zonal_mean` — latitude-band averages
- {py:meth}`uxarray.UxDataArray.azimuthal_mean` — rings of constant great-circle distance
